# 02 — Prétraitement des données

**Objectif** : montrer, étape par étape et chiffres à l'appui, ce que fait le pipeline de
nettoyage (`recsys.preprocessing`) et justifier les seuils d'activité retenus pour le jeu
`interactions` destiné aux modèles.

Le pipeline complet se lance aussi sans notebook : `python scripts/prepare_data.py`.

In [1]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from recsys.analysis import eda, plots
from recsys.analysis.report import compute_statistics
from recsys.config import load_config
from recsys.data.loaders import load_raw_tables, read_header
from recsys.data.schemas import SCHEMAS
from recsys.data.validation import quality_report
from recsys.preprocessing import cleaning
from recsys.preprocessing.pipeline import clean_tables
from recsys.utils.seed import set_seed

%matplotlib inline
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 20)

config = load_config()
set_seed(config.project.seed)


def show(fig):
    """Affiche une figure une seule fois puis libère la mémoire."""
    display(fig)
    plt.close(fig)


print("Dataset :", config.dataset.name, "|", config.raw_dataset_dir)

Dataset : ml-25m | C:\cvp\recommender-system\data\raw\ml-25m


In [2]:
raw = load_raw_tables(config.raw_dataset_dir)
prep = config.preprocessing
{name: len(df) for name, df in raw.items()}

{'ratings': 25000095, 'movies': 62423, 'tags': 1093360, 'links': 62423}

## 1. Films : année, titre, genres

- l'année est extraite de la fin du titre ; absente, elle reste manquante (on n'invente rien) ;
- `(no genres listed)` devient une liste vide ;
- les titres en double portés par des `movieId` différents sont **conservés** : rien ne prouve
  qu'il s'agit du même film, et fusionner des identifiants fausserait les interactions.

In [3]:
movies = cleaning.clean_movies(raw["movies"])
display(movies.head())
pd.Series({
    "films": len(movies),
    "année manquante": int(movies["year"].isna().sum()),
    "sans genre": int((movies["n_genres"] == 0).sum()),
    "année min": int(movies["year"].min()),
    "année max": int(movies["year"].max()),
}).to_frame("valeur")

,movieId,title,title_clean,year,genres,n_genres
0,1,Toy Story (1995),Toy Story,1995,"[Adventure, Animation, Children, Comedy, Fantasy]",5
1,2,Jumanji (1995),Jumanji,1995,"[Adventure, Children, Fantasy]",3
2,3,Grumpier Old Men (1995),Grumpier Old Men,1995,"[Comedy, Romance]",2
3,4,Waiting to Exhale (1995),Waiting to Exhale,1995,"[Comedy, Drama, Romance]",3
4,5,Father of the Bride Part II (1995),Father of the Bride Part II,1995,[Comedy],1


,valeur
films,62423
année manquante,412
sans genre,5062
année min,1874
année max,2019


## 2. Notes : bornes, doublons, horodatage

In [4]:
ratings = cleaning.clean_ratings(raw["ratings"], prep.rating_min, prep.rating_max)
in_catalog = cleaning.restrict_to_catalog(ratings, movies)
display(pd.Series({
    "notes brutes": len(raw["ratings"]),
    "après bornes + doublons": len(ratings),
    "après restriction au catalogue": len(in_catalog),
}).to_frame("lignes"))
ratings = in_catalog
ratings.head()

,lignes
notes brutes,25000095
après bornes + doublons,25000095
après restriction au catalogue,25000095


,userId,movieId,rating,timestamp
0,1,5952,4.000,2006-05-17 12:14:13+00:00
1,1,2012,2.500,2006-05-17 12:14:28+00:00
2,1,2011,2.500,2006-05-17 12:14:39+00:00
3,1,1653,4.000,2006-05-17 12:14:57+00:00
4,1,1250,4.000,2006-05-17 12:20:14+00:00


## 3. Tags : normalisation du texte libre

Espaces superflus retirés, passage en minuscules, tags vides ou manquants écartés, puis
dédoublonnage (utilisateur, film, tag).

In [5]:
tags = cleaning.restrict_to_catalog(cleaning.clean_tags(raw["tags"]), movies)
display(pd.Series({
    "tags bruts": len(raw["tags"]),
    "tags manquants": int(raw["tags"]["tag"].isna().sum()),
    "tags distincts avant normalisation": int(raw["tags"]["tag"].nunique()),
    "tags après nettoyage": len(tags),
    "tags distincts après normalisation": int(tags["tag"].nunique()),
}).to_frame("valeur"))

,valeur
tags bruts,1093360
tags manquants,16
tags distincts avant normalisation,73050
tags après nettoyage,1093333
tags distincts après normalisation,65380


## 4. Liens externes

In [6]:
links = cleaning.restrict_to_catalog(cleaning.clean_links(raw["links"]), movies)
pd.Series({
    "liens": len(links),
    "tmdbId manquants": int(links["tmdbId"].isna().sum()),
    "imdbId manquants": int(links["imdbId"].isna().sum()),
}).to_frame("valeur")

,valeur
liens,62423
tmdbId manquants,107
imdbId manquants,0


## 5. Choix des seuils d'activité

Un film noté une ou deux fois n'offre presque aucun signal collaboratif et rend les métriques
instables. Mais un seuil trop haut ampute le catalogue et aggrave le biais de popularité.
On mesure l'effet de plusieurs seuils (filtrage itératif) avant de trancher.

In [7]:
rows = []
for min_movie in (1, 5, 10, 20, 50, 100):
    kept = cleaning.filter_min_activity(ratings, prep.min_user_ratings, min_movie)
    rows.append({
        "min notes / film": min_movie,
        "notes": len(kept),
        "part des notes": len(kept) / len(ratings),
        "utilisateurs": kept["userId"].nunique(),
        "films": kept["movieId"].nunique(),
        "part des films notés": kept["movieId"].nunique() / ratings["movieId"].nunique(),
        "densité": len(kept) / (kept["userId"].nunique() * kept["movieId"].nunique()),
    })
sensitivity = pd.DataFrame(rows)
sensitivity.style.format({
    "notes": "{:,}", "utilisateurs": "{:,}", "films": "{:,}",
    "part des notes": "{:.2%}", "part des films notés": "{:.2%}", "densité": "{:.3%}",
})

,min notes / film,notes,part des notes,utilisateurs,films,part des films notés,densité
0,1,"25,000,095",100.00%,"162,541","59,047",100.00%,0.260%
1,5,"24,945,390",99.78%,"162,516","32,711",55.40%,0.469%
2,10,"24,889,308",99.56%,"162,473","24,324",41.19%,0.630%
3,20,"24,808,109",99.23%,"162,414","18,424",31.20%,0.829%
4,50,"24,639,412",98.56%,"162,242","13,172",22.31%,1.153%
5,100,"24,434,557",97.74%,"162,065","10,322",17.48%,1.461%


Les films écartés du jeu `interactions` ne disparaissent pas du projet : ils restent dans
`movies.parquet` et seront servis par les approches contenu / embeddings (cold start film).

## 6. Exécution du pipeline et contrôle des sorties

In [8]:
del raw, ratings, tags, links, movies
from recsys.preprocessing.pipeline import run

summary = run(config)
print(json.dumps(summary, indent=2))

20:19:05 | INFO    | recsys.preprocessing.pipeline | Chargement des CSV depuis C:\cvp\recommender-system\data\raw\ml-25m


20:19:47 | INFO    | recsys.preprocessing.pipeline | Nettoyage


20:20:38 | INFO    | recsys.preprocessing.pipeline | movies             62423 lignes -> movies.parquet


20:20:46 | INFO    | recsys.preprocessing.pipeline | ratings         25000095 lignes -> ratings.parquet


20:20:47 | INFO    | recsys.preprocessing.pipeline | tags             1093333 lignes -> tags.parquet


20:20:47 | INFO    | recsys.preprocessing.pipeline | links              62423 lignes -> links.parquet


20:20:56 | INFO    | recsys.preprocessing.pipeline | interactions    24808109 lignes -> interactions.parquet


{
  "ratings": {
    "raw": {
      "n_rows": 25000095,
      "n_users": 162541,
      "n_movies": 59047
    },
    "clean": {
      "n_rows": 25000095,
      "n_users": 162541,
      "n_movies": 59047
    }
  },
  "movies": {
    "raw": {
      "n_rows": 62423,
      "n_movies": 62423
    },
    "clean": {
      "n_rows": 62423,
      "n_movies": 62423
    }
  },
  "tags": {
    "raw": {
      "n_rows": 1093360,
      "n_users": 14592,
      "n_movies": 45251
    },
    "clean": {
      "n_rows": 1093333,
      "n_users": 14591,
      "n_movies": 45251
    }
  },
  "links": {
    "raw": {
      "n_rows": 62423,
      "n_movies": 62423
    },
    "clean": {
      "n_rows": 62423,
      "n_movies": 62423
    }
  },
  "interactions": {
    "n_rows": 24808109,
    "n_users": 162414,
    "n_movies": 18424
  },
  "thresholds": {
    "min_user_ratings": 20,
    "min_movie_ratings": 20
  }
}


In [9]:
rows = []
for path in sorted(config.paths.processed_dir.glob("*.parquet")):
    df = pd.read_parquet(path)
    rows.append({"fichier": path.name, "lignes": len(df), "colonnes": list(df.columns),
                 "taille_Mo": path.stat().st_size / 1e6})
pd.DataFrame(rows)

,fichier,lignes,colonnes,taille_Mo
0,interactions.parquet,24808109,"[userId, movieId, rating, timestamp]",205.098
1,links.parquet,62423,"[movieId, imdbId, tmdbId]",1.294
2,movies.parquet,62423,"[movieId, title, title_clean, year, genres, n_...",2.933
3,ratings.parquet,25000095,"[userId, movieId, rating, timestamp]",208.586
4,tags.parquet,1093333,"[userId, movieId, tag, timestamp]",12.066


In [10]:
interactions = pd.read_parquet(config.paths.processed_dir / "interactions.parquet")
assert not interactions.duplicated(["userId", "movieId"]).any()
assert interactions["rating"].between(prep.rating_min, prep.rating_max).all()
assert interactions["userId"].value_counts().min() >= prep.min_user_ratings
assert interactions["movieId"].value_counts().min() >= prep.min_movie_ratings
assert interactions.notna().all().all()
print("Contrôles passés :", f"{len(interactions):,} interactions prêtes pour les modèles.")

Contrôles passés : 24,808,109 interactions prêtes pour les modèles.
